# C-variance — 같은 코드로 dev 200건 6회

`reports/team-c/c-variance/RUN-CARD.md` 가 소유하는 회차다. **고칠 줄이 없다** —
`REPO_REF` 는 `772ca12` 로 이미 박혀 있고 변동폭 셀도 들어 있다.

위에서부터 순서대로 전부 실행한다. 통과당 약 12분 × 6 = **약 1시간 12분** + 셋업.

기준 노트북(`colab-baseline.ipynb`)에서 dev·dev-debug 검증 통과를 뺐다. 그 둘은
`debug_responses` 가 서로 달라 이 실험의 여섯 통과와 같은 저울이 아니고, 넣으면
24분이 더 든다. 채점은 로컬에서 한다.


In [ ]:
import csv
import gzip
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

WORK = Path(tempfile.mkdtemp(prefix="t1-colab-", dir="/content"))
RESULTS = WORK / "results"
RESULTS.mkdir()
PYTHON = str(WORK / "venv/bin/python")
MODEL_ID = "google/gemma-4-26B-A4B-it"
REVISION = "4d7ae4984b7db7de8f8457170b3f1a419ee76d52"
EXPECTED_PACKAGES = {"vllm": "0.26.0", "torch": "2.11.0+cu130",
                     "transformers": "5.14.1", "xgrammar": "0.2.3"}
SERVER_PYTHON = "3.12.13"
case_inputs = {}
SOURCE_MODE = "clone"  # 특정 로컬 ZIP을 검사하려면 "upload"
REPO_URL = "https://github.com/LittleBitAI/ai-nara-shop.git"
REPO_REF = "772ca1284c8918c1fa4ca5729cc15dfe4dedb2d4"  # C-variance: 코드를 한 글자도 안 바꾸는 것이 이 실험의 전부다

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n",
                    encoding="utf-8", newline="\n")

def run_logged(name, command, env=None, cwd=None):
    started = time.time()
    entry = {"argv": [str(x) for x in command], "cwd": str(cwd or WORK), "started": started,
             "returncode": None}
    if entry["argv"][0] == PYTHON:
        # Absolute Python paths do not activate PATH for tools such as ninja.
        env = dict(os.environ if env is None else env)
        env["PATH"] = str(Path(PYTHON).parent) + os.pathsep + env.get("PATH", "")
        env["VIRTUAL_ENV"] = str(Path(PYTHON).parent.parent)
    log_path = RESULTS / (name + ".log")
    record_path = RESULTS / (name + "-command.json")
    if log_path.exists() or record_path.exists():
        raise ValueError(f"{name} 실행 기록이 이미 있습니다. 첫 셀부터 새 작업 폴더로 실행하세요.")
    write_json(record_path, entry)
    try:
        with log_path.open("x", encoding="utf-8", newline="\n") as log_file:
            with subprocess.Popen(entry["argv"], cwd=entry["cwd"], env=env, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                  errors="replace", bufsize=1) as process:
                try:
                    for line in process.stdout:
                        log_file.write(line)
                        log_file.flush()
                        print(line, end="")
                    entry["returncode"] = process.wait()
                except BaseException:
                    process.terminate()
                    try:
                        process.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        process.kill()
                    raise
    finally:
        entry["elapsed_seconds"] = time.time() - started
        write_json(record_path, entry)
    if entry["returncode"] != 0:
        raise RuntimeError(f"{name} 실패 (exit={entry['returncode']}). 마지막 로그 다운로드 셀을 실행하세요.")
    return log_path

write_json(RESULTS / "host.json", {"python": sys.version, "work": str(WORK)})
print("작업 폴더:", WORK)


## 1. git clone으로 코드·데이터와 제출 ZIP 준비

기본은 공개 저장소 clone입니다. 위 설정의 `REPO_REF`는 기본 `main`이며 실제 받은 커밋 SHA를 기록합니다.
재현할 때는 그 SHA를 지정하세요. clone한 코드의 기존 패키징 도구로 제출 ZIP과 검증 번들을 만듭니다.
아래 ZIP 검사는 자동 생성된 번들을 읽으며 수동 업로드가 필요 없습니다.

이미 만든 로컬 ZIP을 검사할 경우에만 첫 셀의 `SOURCE_MODE = "upload"`로 바꿉니다.
이때 로컬에서 만든 `colab-bundle.zip` 하나를 업로드합니다.


In [ ]:
if SOURCE_MODE == "clone":
    REPO = WORK / "repo"
    BUNDLE_PATH = WORK / "candidate-colab-bundle.zip"
    run_logged("git-clone", ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO)])
    if REPO_REF != "main":
        # GitHub은 임의 SHA의 fetch를 40자 전체로만 받습니다. 약칭은 remote ref가 아니므로
        # `fatal: couldn't find remote ref`로 죽습니다. 브랜치 이름은 그대로 됩니다.
        if 7 <= len(REPO_REF) < 40 and all(c in "0123456789abcdef" for c in REPO_REF):
            raise ValueError(f'REPO_REF="{REPO_REF}"는 약칭 SHA입니다. 40자 전체 SHA나 브랜치 이름을 쓰세요.')
        run_logged("git-fetch-ref", ["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REPO_REF])
        run_logged("git-checkout-ref", ["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"])
    commit_log = run_logged("git-commit", ["git", "-C", str(REPO), "rev-parse", "HEAD"])
    SOURCE_COMMIT = commit_log.read_text(encoding="utf-8").strip()
    write_json(RESULTS / "source.json", {"mode": "clone", "url": REPO_URL,
                                      "requested_ref": REPO_REF, "commit": SOURCE_COMMIT})
    run_logged("package", [sys.executable, "-X", "utf8", str(REPO / "tools/package.py"),
        "--output", str(WORK / "candidate-submit.zip"), "--colab-output", str(BUNDLE_PATH)], cwd=REPO)
    print("검증할 저장소 커밋:", SOURCE_COMMIT)
elif SOURCE_MODE != "upload":
    raise ValueError('SOURCE_MODE는 "clone" 또는 "upload"여야 합니다.')


In [ ]:
if SOURCE_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("colab-bundle.zip 한 개만 선택하세요.")
    bundle_bytes = next(iter(uploaded.values()))
    del uploaded
    write_json(RESULTS / "source.json", {"mode": "upload"})
else:
    bundle_bytes = BUNDLE_PATH.read_bytes()
# tools/package.py의 COLAB_FILES와 같아야 합니다. 한쪽만 고치면 이 검사가 막습니다.
allowed = {"submit.zip", "tools/score.py", "tools/diagnose_items.py",
           "open/dev.jsonl", "open/dev_labels.csv",
           "open/data/test.jsonl.gz", "open/data/항목표.json", "open/data/정답스키마_디코딩.json",
           "open/data/법령패키지/법령/중소기업제품 구매촉진 및 판로지원에 관한 법률.txt",
           "open/data/법령패키지/법령/중소기업제품 구매촉진 및 판로지원에 관한 법률 시행령.txt",
           "open/data/법령패키지/중기부고시/중기부고시_경쟁제품_세부품명.csv",
           "bundle-manifest.json"}
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    if set(archive.namelist()) != allowed or len(archive.namelist()) != len(allowed):
        raise ValueError("Colab 번들의 파일 목록이 다릅니다.")
    if sum(info.file_size for info in archive.infolist()) > 250_000_000:
        raise ValueError("Colab 번들이 예상 크기를 초과합니다.")
    manifest = json.loads(archive.read("bundle-manifest.json"))
    if set(manifest["sha256"]) != allowed - {"bundle-manifest.json"}:
        raise ValueError("번들 해시 목록이 다릅니다.")
    contents = {name: archive.read(name) for name in allowed}
    for name, expected in manifest["sha256"].items():
        if hashlib.sha256(contents[name]).hexdigest() != expected:
            raise ValueError(f"번들 해시 불일치: {name}")
    for name, content in contents.items():
        destination = WORK / name  # exact allowlist checked above
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)

SUBMISSION = WORK / "submission"
SUBMISSION.mkdir()
with zipfile.ZipFile(WORK / "submit.zip") as archive:
    if set(archive.namelist()) != {"script.py", "requirements.txt"} or len(archive.namelist()) != 2:
        raise ValueError("제출 ZIP 루트가 두 파일이 아닙니다.")
    if sum(info.file_size for info in archive.infolist()) > 10_000_000:
        raise ValueError("제출 ZIP이 예상 크기를 초과합니다.")
    for name in archive.namelist():
        (SUBMISSION / name).write_bytes(archive.read(name))
SCRIPT_SHA256 = hashlib.sha256((SUBMISSION / "script.py").read_bytes()).hexdigest()
write_json(RESULTS / "bundle-manifest.json", manifest)
write_json(RESULTS / "candidate.json", {
    "bundle_sha256": hashlib.sha256(bundle_bytes).hexdigest(),
    "submit_sha256": manifest["sha256"]["submit.zip"], "script_sha256": SCRIPT_SHA256,
    "model_id": MODEL_ID, "revision": REVISION,
})
print("제출 ZIP SHA-256:", manifest["sha256"]["submit.zip"])
print("실행 script.py SHA-256:", SCRIPT_SHA256)
del bundle_bytes, contents


## 2. 실제 GPU와 디스크 확인

GPU 종류·드라이버·VRAM·RAM을 기록합니다. 30GiB VRAM/80GiB 여유 디스크는 이 노트북의 사전 거름 기준이며 적재 보장이 아닙니다. 부족하면 모델 다운로드 전에 멈춥니다.


In [ ]:
gpu_log = run_logged("gpu", ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                             "--format=csv,noheader"])
gpus = gpu_log.read_text(encoding="utf-8").strip().splitlines()
if not gpus or max(int(row.split(",")[1].strip().split()[0]) for row in gpus) < 30 * 1024:
    raise RuntimeError("VRAM 30GiB 미만입니다. 더 큰 GPU 런타임을 선택한 뒤 처음부터 실행하세요.")
resources = {"gpus": gpus, "disk_free_bytes": shutil.disk_usage(WORK).free,
             "meminfo": Path("/proc/meminfo").read_text()}
write_json(RESULTS / "resources.json", resources)
if resources["disk_free_bytes"] < 80 * 1024**3:
    raise RuntimeError("설치와 원본 모델을 준비할 여유 디스크 80GiB가 필요합니다.")
print("GPU 사전 확인 통과. 실제 적재/실행 성공은 아직 확인하지 않았습니다.")


## 3. 대회 Python·패키지·설치 경로 재현

uv는 정확한 Python 3.12.13을 준비하는 Colab 도구로만 설치합니다. 추론 패키지는 별도 venv에 고정 버전으로 설치합니다.
이후 제출 ZIP의 requirements.txt를 설치하고, 실제 Python·핵심 패키지·CUDA 빌드가 명세와 다르면 중단합니다.

[대회 서버 명세](https://www.dacon.io/competitions/official/236754/overview/evaluation) · [uv Python 관리](https://docs.astral.sh/uv/guides/install-python/).
서버 컨테이너 이미지 전체나 호스트 GPU/드라이버까지 동일하게 복제하는 것은 아닙니다.


In [ ]:
# Colab 커널 Python과 분리해 대회 서버의 정확한 Python 버전을 준비합니다.
run_logged("uv-install", [sys.executable, "-m", "pip", "install", "uv"])
run_logged("python-install", [sys.executable, "-m", "uv", "venv", "--python", SERVER_PYTHON,
                             "--seed", str(WORK / "venv")])
run_logged("install", [PYTHON, "-m", "pip", "install",
    "--extra-index-url", "https://download.pytorch.org/whl/cu130",
    *[f"{name}=={version}" for name, version in EXPECTED_PACKAGES.items()]])
# 서버처럼 제출 ZIP 안의 requirements.txt도 설치합니다.
run_logged("submission-install", [PYTHON, "-m", "pip", "install", "-r",
                                  str(SUBMISSION / "requirements.txt")])
if json.loads((RESULTS / "submission-install-command.json").read_text())["elapsed_seconds"] > 600:
    raise RuntimeError("제출 requirements 설치가 서버 제한 600초를 넘었습니다.")
run_logged("pip-freeze", [PYTHON, "-m", "pip", "freeze"])
runtime_code = """
import json, platform, shutil, subprocess, sys, torch, vllm, transformers, xgrammar
from importlib.metadata import version
from pathlib import Path
runtime = {"python": platform.python_version(), "cuda": torch.version.cuda,
           "packages": {n: version(n) for n in ["torch", "vllm", "transformers", "xgrammar"]},
           "gpu": torch.cuda.get_device_name(0),
           "ninja_path": shutil.which("ninja"),
           "ninja_version": subprocess.check_output(["ninja", "--version"], text=True).strip()}
Path(sys.argv[1]).write_text(json.dumps(runtime, indent=2) + "\\n", encoding="utf-8")
print(runtime)
torch.empty(1, device="cuda")
"""
run_logged("runtime", [PYTHON, "-c", runtime_code, str(RESULTS / "runtime.json")])
runtime = json.loads((RESULTS / "runtime.json").read_text())
if (runtime["python"] != SERVER_PYTHON or runtime["packages"] != EXPECTED_PACKAGES
        or runtime["cuda"] != "13.0"):
    raise RuntimeError("서버의 Python/핵심 패키지/CUDA 빌드와 다릅니다. 검증을 중단합니다.")


## 4. HF_TOKEN으로 고정 리비전 모델 다운로드

Colab 보안 비밀에 `HF_TOKEN`을 등록하고 이 노트북의 접근을 허용하세요. 읽지 못하면 숨김 입력으로 받으며 **빈 토큰은 거부**합니다.
토큰은 모델 다운로드 자식 프로세스 환경변수로만 전달하고, 추론 환경·명령행·로그·결과 ZIP에 기록하지 않습니다.

[고정 모델](https://huggingface.co/google/gemma-4-26B-A4B-it/tree/4d7ae4984b7db7de8f8457170b3f1a419ee76d52)의 접근 권한이 있는 계정을 사용합니다.
가중치는 준비 단계에서만 내려받고 제출물에 포함하지 않습니다.


In [ ]:
from google.colab import userdata
from getpass import getpass

try:
    token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = getpass("HF_TOKEN (Hugging Face 읽기 토큰, 필수): ")
if not token or not token.strip():
    raise ValueError("HF_TOKEN이 필요합니다. Colab 보안 비밀에 등록하고 노트북 접근을 허용하세요.")
download_env = {**os.environ, "HF_TOKEN": token.strip()}
download_env.pop("HF_HUB_OFFLINE", None)
download_env.pop("TRANSFORMERS_OFFLINE", None)
download_code = """
from huggingface_hub import snapshot_download
from pathlib import Path
import os
import sys
model_dir = snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], token=os.environ["HF_TOKEN"],
    allow_patterns=["*.safetensors", "*.json", "*.jinja", "*.model"])
if Path(model_dir).name != sys.argv[2]:
    raise RuntimeError("고정 snapshot 경로 불일치")
Path(sys.argv[3]).write_text(model_dir, encoding="utf-8")
"""
try:
    run_logged("model-download", [PYTHON, "-c", download_code, MODEL_ID, REVISION,
                                 str(WORK / "model-path.txt")], env=download_env)
finally:
    download_env.pop("HF_TOKEN", None)
    token = None
MODEL_DIR = (WORK / "model-path.txt").read_text(encoding="utf-8")
write_json(RESULTS / "model.json", {"id": MODEL_ID, "revision": REVISION, "path": MODEL_DIR})


## 5. 서버 진입점으로 샘플 10건 실행

제출 ZIP 코드를 별도 디렉터리에 놓고 `python script.py`로 실행합니다. 경로만 PPS 환경변수로 지정합니다.
양자화·문맥·출력 예산·청크·seed 등은 제출 코드 기본값을 쓰고, 성공 보고서의 실제 설정도 확인합니다.
`--debug-responses`·`--mock`·Colab 전용 추론 옵션을 사용하지 않습니다. 실패 시 마지막 로그 다운로드 셀로 이동합니다.


In [ ]:
def run_case(name, source, args=()):
    case = WORK / "cases" / name
    data = case / "data"
    data.mkdir(parents=True)
    for filename in ("script.py", "requirements.txt"):
        shutil.copyfile(SUBMISSION / filename, case / filename)
    for filename in ("항목표.json", "정답스키마_디코딩.json"):
        shutil.copyfile(WORK / "open/data" / filename, data / filename)
    shutil.copytree(WORK / "open/data/법령패키지", data / "법령패키지")
    # dev도 서버와 같은 test.jsonl.gz 입력 경로를 거칩니다.
    if source.suffix == ".gz":
        shutil.copyfile(source, data / "test.jsonl.gz")
    else:
        # mtime=0으로 고정해야 같은 dev.jsonl이 회차마다 같은 .gz 바이트가 됩니다.
        # gzip 기본값은 현재 시각을 헤더에 넣어 input_sha256이 매번 달라집니다.
        with source.open("rb") as original, (data / "test.jsonl.gz").open("wb") as raw:
            with gzip.GzipFile(fileobj=raw, mode="wb", mtime=0) as compressed:
                shutil.copyfileobj(original, compressed)
    case_inputs[name] = hashlib.sha256((data / "test.jsonl.gz").read_bytes()).hexdigest()
    env = {key: value for key, value in os.environ.items()
           if not key.startswith(("PPS_", "VLLM_")) and key not in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")}
    env.update(PPS_MODEL_DIR=MODEL_DIR, PPS_DATA_DIR=str(data), PPS_OUTPUT_DIR=str(RESULTS / name),
               HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONUNBUFFERED="1",
               CUDA_VISIBLE_DEVICES="0", VLLM_NO_USAGE_STATS="1", HF_HUB_DISABLE_TELEMETRY="1")
    # 검증 회차는 인자를 넘기지 않습니다. check_live가 무인자 실행을 다시 확인합니다.
    # args는 진단 전용 회차(--debug-responses)에만 씁니다. 그 회차는 검증 통과로 세지 않습니다.
    run_logged(name, [PYTHON, "script.py", *args], env=env, cwd=case)

def check_live(name, expected):
    report = json.loads((RESULTS / name / "run_report.json").read_text(encoding="utf-8"))
    if (report["mode"] != "live" or report["model_success_count"] != expected
            or report.get("sme_model_success_count") != report.get("sme_verified_count")
            or report.get("sme_verified_count", -1) < 0 or report.get("sme_fallback_count", -1) < 0
            or report.get("sme_verified_count", -1) + report.get("sme_fallback_count", -1) != report.get("sme_selected_count")
            or report.get("sme_skipped_count", -1) < 0
            or report.get("sme_selected_count", -1) + report.get("sme_skipped_count", -1) != expected
            or report["건수"] != expected or report["자가검증"] != "PASS"
            or report["code_sha256"] != SCRIPT_SHA256):
        raise RuntimeError("실제 모델 성공 건수/코드/CSV 검사 불일치")
    if report["sme_fallback_count"]:
        print(f"{name}: 추가 분석 실패 {report['sme_fallback_count']}건은 검증된 기본 모델 판정을 보존했습니다.")
    reproduction = report["reproduction"]
    settings = reproduction["settings"]
    expected_settings = {"quant": "int8_per_channel_weight_only", "max_model_len": 16384,
        "max_tokens": 2048, "max_chars": 16000, "chunk": 128, "gpu_mem": 0.92, "tp": 1,
        "seed": 20260826, "temperature": 0, "thinking": False, "limit": None, "debug_responses": False,
        "sme_items": ["v13"], "sme_selection": "baseline_v13_positive",
        "prompt_language": "en_with_ko_legal_terms", "sme_facts": True}
    if any(settings.get(key) != value for key, value in expected_settings.items()):
        raise RuntimeError("서버 기본 추론 설정 불일치")
    if (reproduction["python"] != SERVER_PYTHON
            or any(reproduction["packages"].get(key) != value for key, value in EXPECTED_PACKAGES.items())
            or report["environment"].get("cuda") != "13.0"):
        raise RuntimeError("서버 Python/패키지/CUDA 빌드 불일치")
    # 기록된 model_dir은 개인 절대 경로를 남기지 않으므로 고정 리비전 이름으로 대조합니다.
    if (report["model"] != {"id": MODEL_ID, "expected_revision": REVISION}
            or Path(settings["model_dir"]).name != REVISION or Path(MODEL_DIR).name != REVISION
            or report["input_sha256"] != case_inputs[name]):
        raise RuntimeError("모델 리비전/경로 또는 실행 입력 불일치")
    for path, expected_hash in manifest["sha256"].items():
        if path.startswith("open/data/법령패키지/"):
            if reproduction["asset_sha256"].get(path.removeprefix("open/data/")) != expected_hash:
                raise RuntimeError("실행에 사용한 법령/품목 파일 해시 불일치")
    command = json.loads((RESULTS / (name + "-command.json")).read_text())
    if command["returncode"] != 0 or command["argv"] != [PYTHON, "script.py"]:
        raise RuntimeError("서버 무인자 실행 명령/종료 코드 불일치")
    if command["elapsed_seconds"] > 7200:
        raise RuntimeError("실행이 서버 제한 7200초를 넘었습니다.")
    # 실행 후에도 내보낼 ZIP과 실제 실행 파일이 같은지 확인합니다.
    columns = ["id"] + [f"v{i}" for i in range(1, 25)] + [f"e{i}" for i in range(1, 25)]
    paired = []
    for filename in ("baseline_submission.csv", "submission.csv"):
        with (RESULTS / name / filename).open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != columns:
                raise RuntimeError("비교 CSV 헤더 불일치")
            paired.append(list(reader))
    if any(len(rows) != expected for rows in paired):
        raise RuntimeError("비교 CSV 건수 불일치")
    extra_owned = set()
    for phase_items in (settings.get("extra_call_items") or {}).values():
        extra_owned |= set(phase_items)
    owned = set(settings["sme_items"]) | extra_owned
    allowed = owned | {"e" + item[1:] for item in owned}
    protected = set(columns) - allowed
    if any(before[key] != after[key] for before, after in zip(*paired) for key in protected):
        raise RuntimeError("추가 호출이 대상 밖 항목/근거 또는 ID를 변경했습니다.")
    # 선택은 후처리 **전** 원판정의 v13 양성으로 한다(`baseline_v13_positive`).
    # `baseline_submission.csv`는 후처리를 거친 뒤이고, 근거 계약이 검증된 인용 없는 양성을
    # 내리므로 그 v13 양성은 선택 집합의 **부분집합**이다. 같기를 요구하면 계약이 한 건이라도
    # 일하는 순간 빨개진다 — dev 200건에서 실제로 104건 중 25건이 내려간다.
    # 지켜야 할 것은 포함 관계다. 살아남은 양성이 선택보다 많으면 추가 분석이 그 건을 못 봤다.
    if sum(row["v13"] == "1" for row in paired[0]) > report["sme_selected_count"]:
        raise RuntimeError("v13 추가 분석이 기본 판정 양성 일부를 선택하지 않았습니다")
    # SME 생략은 다른 단계의 변경 권한을 취소하지 않는다. A1도 v13을 소유한다.
    skipped_protected = {key for item in set(settings["sme_items"]) - extra_owned
                         for key in (item, "e" + item[1:])}
    if any(before[key] != after[key] for before, after in zip(*paired)
           if before["v13"] == "0" for key in skipped_protected):
        raise RuntimeError("추가 분석을 생략한 공고가 변경됐습니다.")
    with zipfile.ZipFile(WORK / "submit.zip") as archive:
        for filename in ("script.py", "requirements.txt"):
            if archive.read(filename) != (WORK / "cases" / name / filename).read_bytes():
                raise RuntimeError("검증 중 실행 파일이 제출 ZIP과 달라졌습니다.")
    return report

run_case("sample", WORK / "open/data/test.jsonl.gz")
sample_report = check_live("sample", 10)
print("샘플 10건: 실제 모델·서버 진입점·설정 검사 통과. 다음은 dev 전체 검사입니다.")


## 6. 같은 코드로 dev 200건을 6회

여섯 통과가 **전부 같은 설정**이다. `--debug-responses` 를 켜는 이유는
`README.md` §3-4 에 있다 — 판정 경로에 영향이 없음을 코드로 확인했고(§1-2),
원응답이 남으면 이 비싼 회차를 나중에 재생 실험에 다시 쓸 수 있다.

`check_live` 는 여기서 **안 쓴다.** 그 함수는 `debug_responses=False` 를 요구한다.
대신 아래 셀이 통과마다 자체 검사를 한다.


In [ ]:
# C-variance: 같은 코드·같은 입력·같은 설정으로 dev 200건을 6회 돈다.
VARIANCE_N = 6
VARIANCE_ARGS = ["--debug-responses"]   # 여섯 통과 전부 같은 값

for k in range(1, VARIANCE_N + 1):
    name = f"var-{k:02d}"
    run_case(name, WORK / "open/dev.jsonl", args=VARIANCE_ARGS)
    report = json.loads((RESULTS / name / "run_report.json").read_text(encoding="utf-8"))
    settings = report["reproduction"]["settings"]
    if report["mode"] != "live" or not settings["debug_responses"]:
        raise RuntimeError(f"{name}: live 가 아니거나 원응답을 안 켰다")
    if report["건수"] != 200 or report["model_success_count"] != 200:
        raise RuntimeError(f"{name}: 200건 전건 성공이 아니다")
    if report["code_sha256"] != SCRIPT_SHA256:
        raise RuntimeError(f"{name}: 코드 해시가 다르다")
    if report["input_sha256"] != case_inputs[name]:
        raise RuntimeError(f"{name}: 입력 해시가 다르다")
    if report["sme_fallback_count"] or report["company_size_fallback_count"]:
        raise RuntimeError(f"{name}: fallback 이 있다 — 이 통과는 버린다")
    events = [json.loads(line) for line
              in (RESULTS / name / "diagnostics.jsonl").read_text(encoding="utf-8").splitlines()]
    responses = [e for e in events if e["event"] == "response"]
    if not responses or any("response_text" not in e for e in responses):
        raise RuntimeError(f"{name}: 원응답이 안 남았다")
    print(f"{name}: 전건 성공 · 추론 {report['추론_s']}s · 원응답 {len(responses)}건")

print()
print(f"{VARIANCE_N}회 완료. 채점은 로컬에서 한다 — 노트북 점수를 판정에 쓰지 않는다.")


## 7. 결과 ZIP 을 받는다

**중간에 무엇이 실패했어도 이 셀은 따로 실행한다** — 추론이 끝난 통과의 자료는
이미 `RESULTS` 에 있다(`docs/colab.md` §8).


In [ ]:
from google.colab import files

archive_path = WORK / ("colab-results-" + str(time.time_ns()) + ".zip")
with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(RESULTS).as_posix())
print("검증 결과:", archive_path.name)
files.download(str(archive_path))
